# 階段⑥-A 擴量:重編房源向量 + recall gate — Colab GPU

擴量後房源 704 → **974**(`property_data.json` 已含),但 `property_embeddings.json` 仍 704。
本 notebook 用 **rbt3 bi-encoder student** 重編全部 974 房源向量,再跑 recall gate,
通過就下載 `property_embeddings.json` 覆蓋 → 整批入庫。

**先做**:Runtime → Change runtime type → **GPU**(T4 即可)。

**同源關鍵**:現役前端 `bi_encoder_quant.onnx` 是 rbt3 student(#78 蒸餾)。
房源向量必須用**同一顆 rbt3** 重編,否則 query 向量(前端)與房源向量不同空間 → 召回壞。
- 路徑 1A:Drive 有存 #78 student → 直接用(與現役 onnx 同源,最省)。
- 路徑 1B:沒存 → 重蒸一顆 rbt3(需把 onnx 也一起 export 覆蓋,gate 才同源)。

分支:`claude/stage6-expansion`(已含 974 筆 property_data.json)。

## 0. 環境 — clone 擴量分支 + 裝套件

In [ ]:
!nvidia-smi -L
%cd /content
![ -d nchu-edge-rental-ai ] || git clone https://github.com/eric20041027/nchu-edge-rental-ai.git
%cd /content/nchu-edge-rental-ai
!git fetch origin && git checkout claude/stage6-expansion && git pull
!pip -q install 'transformers<5' tokenizers onnx onnxruntime
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
# 確認分支已含 974 筆
import json; print('property_data.json:', len(json.load(open('frontend/assets/property_data.json'))), '筆(應 974)')

## 1. 取得 rbt3 student 權重

兩條路,擇一:
- **(A) Drive 已存**:上次蒸餾(#78)有存 `saved_models/rbt3_bi_encoder` 到 Drive → 掛 Drive 複製過來。
- **(B) 沒存 → 重蒸**:跑下面〈1B〉重蒸一顆 rbt3 student(從 hfl/rbt6 teacher,~10 分鐘)。

重蒸的 student 與現役 onnx 的差異:同架構(rbt3）、同訓練法,recall gate 會驗證等效。

### 1A. (若 Drive 有存)掛 Drive 複製 student

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# ↓ 改成你 Drive 裡 rbt3_bi_encoder 的實際路徑(含 config.json + model.safetensors)
DRIVE_STUDENT = '/content/drive/MyDrive/rbt3_bi_encoder'
import os, shutil
assert os.path.exists(f'{DRIVE_STUDENT}/config.json'), '找不到 student 權重,改走 1B 重蒸'
shutil.copytree(DRIVE_STUDENT, 'saved_models/rbt3_bi_encoder', dirs_exist_ok=True)
print('✓ student 已複製到 saved_models/rbt3_bi_encoder')

### 1B. (若沒存)重蒸 rbt3 student

先訓 rbt6 teacher,再蒸 rbt3。詳見 `docs/spec/bi-encoder-distill.md`。

In [ ]:
# teacher(rbt6)
!python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32
# 蒸餾 → rbt3 student
!python -m pipeline.model_training.distill_bi_encoder \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32

## 2. 用 rbt3 student 重編 974 房源向量

輸出 `frontend/assets/property_embeddings.json`(974 × 768,float16）。
與 `property_data.json` 同批、同源。

In [ ]:
!python -m pipeline.data_prep.build_property_embeddings \
    --saved-dir saved_models/rbt3_bi_encoder
!python -m pipeline.data_prep.build_property_embeddings --check
# --check 應顯示 property_data.json 974 records、property_embeddings 974 vecs in sync

## 3. Recall gate — 擴量後召回沒崩

讀重編的向量 + 前端 onnx,跑 vector vs rule-based A/B。
704→974 擴量後 recall@K 應守住(仍 GO);若崩(NO-GO/exit 2)→ 不入庫,查原因。

In [ ]:
# gate 讀前端 onnx(bi_encoder_quant.onnx)當 query encoder,須與重編向量同源。
#   路徑 1A(Drive = #78 student,就是現役 onnx 的來源)→ onnx 已同源,RE_EXPORT=False。
#   路徑 1B(重蒸新 student)→ 必須 export 覆蓋前端 onnx,gate 才同源,RE_EXPORT=True。
RE_EXPORT = False   # ← 若走 1B 重蒸,改成 True

if RE_EXPORT:
    !python -m pipeline.model_training.export_bi_encoder --saved-dir saved_models/rbt3_bi_encoder
!python tests/eval_vector_vs_rulebased.py
# verdict GO → exit 0;NO-GO → exit 2

## 4. Gate 過 → 打包下載入庫檔

下載**兩個**(若步驟 1 重蒸則三個,含 onnx):
- `property_embeddings.json`(974 向量,必下)
- 若重蒸:`bi_encoder_quant.onnx`(新 student onnx,與向量同源,必一起下)

In [ ]:
import json, os
emb = json.load(open('frontend/assets/property_embeddings.json'))
print(f"property_embeddings.json: {emb['count']} vecs x {emb['dim']}d (應 974 x 768)")
assert emb['count'] == len(json.load(open('frontend/assets/property_data.json'))), '向量數 != 房源數!'

# 打包:向量必下;onnx 只在 1B 重蒸(RE_EXPORT=True)時才一起帶(新 student onnx)。
# 1A 路徑 onnx 沒變,不需重帶。
files = 'frontend/assets/property_embeddings.json'
if RE_EXPORT:
    files += ' frontend/models/bi_encoder_dir/bi_encoder_quant.onnx'
!cd /content/nchu-edge-rental-ai && tar czf /content/expansion_artifacts.tgz {files}
from google.colab import files as dl
dl.download('/content/expansion_artifacts.tgz')

## 落地(本機)

```bash
cd <repo>  # claude/stage6-expansion 分支
tar xzf ~/Downloads/expansion_artifacts.tgz   # 覆蓋向量(+ 若重蒸的 onnx)
git add frontend/assets/property_embeddings.json
# 若含新 onnx 也 add:git add frontend/models/bi_encoder_dir/bi_encoder_quant.onnx
git commit -m 'data(stage6-A): 重編 974 房源向量,擴量整批入庫'
```

⚠️ `property_embeddings.json` 與 `property_data.json` **必須同為 974**(--check 已驗)。
落地後此分支才可 merge(前端 974 房源 ↔ 974 向量同步)。冷載入會隨房源增多略增,
用 `benchmark.html` 量。